In [2]:
# ==========================================
# DAY 1 - Haystack Pipeline Architecture
# ==========================================

!pip install -q haystack-ai

from haystack import Pipeline, Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

# Create documents
documents = [
    Document(content="Python is a programming language used for AI and data science."),
    Document(content="Haystack is a framework for building RAG applications."),
    Document(content="RAG means Retrieval Augmented Generation.")
]

# Create document store
document_store = InMemoryDocumentStore()

# Store documents
document_store.write_documents(documents)

# Create retriever
retriever = InMemoryBM25Retriever(
    document_store=document_store
)

# Create pipeline
pipeline = Pipeline()

# Add retriever to pipeline
pipeline.add_component("retriever", retriever)

# Connect pipeline
# pipeline.connect("retriever.documents", "retriever.documents") # This line caused the error

print("Haystack Pipeline created successfully!")
print("Number of documents:", len(documents))

# Search
result = retriever.run(query="What is RAG?")

print("\nRetrieved Documents:")
for doc in result["documents"]:
    print("-", doc.content)

Haystack Pipeline created successfully!
Number of documents: 3

Retrieved Documents:
- Haystack is a framework for building RAG applications.
- RAG means Retrieval Augmented Generation.
- Python is a programming language used for AI and data science.


In [3]:
# ==========================================
# DAY 2 - Haystack BM25 Retrieval
# ==========================================

!pip install -q haystack-ai

from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

# Create documents
documents = [
    Document(content="Python is used for machine learning and artificial intelligence."),
    Document(content="JavaScript is mainly used for web development."),
    Document(content="Machine learning allows computers to learn from data."),
    Document(content="RAG combines document retrieval with large language models."),
    Document(content="Haystack is useful for building search and RAG applications.")
]

# Create document store
document_store = InMemoryDocumentStore()

# Add documents
document_store.write_documents(documents)

# Create BM25 retriever
retriever = InMemoryBM25Retriever(
    document_store=document_store,
    top_k=3
)

# Search query
query = "machine learning"

result = retriever.run(query=query)

print("Query:", query)
print("\nTop matching documents:\n")

for i, doc in enumerate(result["documents"], 1):
    print(f"{i}. {doc.content}")

# Try another query
query = "RAG"

result = retriever.run(query=query)

print("\n\nQuery:", query)
print("\nTop matching documents:\n")

for i, doc in enumerate(result["documents"], 1):
    print(f"{i}. {doc.content}")

Query: machine learning

Top matching documents:

1. Machine learning allows computers to learn from data.
2. Python is used for machine learning and artificial intelligence.
3. JavaScript is mainly used for web development.


Query: RAG

Top matching documents:

1. RAG combines document retrieval with large language models.
2. Haystack is useful for building search and RAG applications.
3. Python is used for machine learning and artificial intelligence.


In [1]:
# ==========================================
# DAY 3 - LlamaIndex Document Indexing
# NO OPENAI API KEY REQUIRED
# ==========================================

!pip install -q llama-index-core llama-index-embeddings-huggingface

# Import
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# ------------------------------------------------
# IMPORTANT:
# Disable the default OpenAI LLM
# ------------------------------------------------

Settings.llm = None

# Use a free HuggingFace embedding model
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

# ------------------------------------------------
# Create documents
# ------------------------------------------------

documents = [
    Document(
        text="Python is a programming language used for artificial intelligence and data science."
    ),

    Document(
        text="RAG stands for Retrieval Augmented Generation. "
             "RAG retrieves information from documents."
    ),

    Document(
        text="LlamaIndex is a framework for working with documents "
             "and large language model applications."
    ),

    Document(
        text="Haystack is a framework for building search and RAG applications."
    )
]

print("Documents created:", len(documents))

# ------------------------------------------------
# Create vector index
# ------------------------------------------------

index = VectorStoreIndex.from_documents(documents)

print("Vector index created successfully!")

# ------------------------------------------------
# Create retriever
# ------------------------------------------------

retriever = index.as_retriever(similarity_top_k=2)

# ------------------------------------------------
# Search
# ------------------------------------------------

question = "What is RAG?"

results = retriever.retrieve(question)

print("\nQuestion:", question)

print("\nRelevant Documents:")

for i, result in enumerate(results, 1):
    print(f"\n{i}.")
    print(result.text)

print("\n--------------------------------")
print("Day 3 completed successfully!")
print("--------------------------------")

LLM is explicitly disabled. Using MockLLM.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Documents created: 4
Vector index created successfully!

Question: What is RAG?

Relevant Documents:

1.
RAG stands for Retrieval Augmented Generation. RAG retrieves information from documents.

2.
Haystack is a framework for building search and RAG applications.

--------------------------------
Day 3 completed successfully!
--------------------------------


In [3]:
# ==========================================
# DAY 4 - LlamaIndex + Ollama Local RAG
# ==========================================

# Install packages
!pip install -q llama-index llama-index-llms-ollama

# Install zstd dependency for Ollama
!apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# Start Ollama
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

# Download a small model
!ollama pull llama3.2:1b

# Import LlamaIndex
from llama_index.core import VectorStoreIndex, Document
from llama_index.llms.ollama import Ollama

# Connect LlamaIndex to Ollama
llm = Ollama(
    model="llama3.2:1b",
    request_timeout=120.0
)

# Create documents
document_list = [
    Document(
        text="Haystack is a framework for building search and RAG applications."
    ),

    Document(
        text="LlamaIndex helps connect large language models with external data."
    ),

    Document(
        text="RAG stands for Retrieval Augmented Generation. "
             "RAG retrieves relevant information from documents and "
             "uses that information to generate answers."
    )
]

# Create index
index = VectorStoreIndex.from_documents(document_list)

# Create query engine using Ollama
query_engine = index.as_query_engine(llm=llm)

# Ask question
question = "What is RAG?"

response = query_engine.query(question)

print("Question:", question)
print("\nAnswer:")
print(response)

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (655 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding

In [4]:
# ==========================================
# DAY 5 - Multi-Document RAG System
# ==========================================

!pip install -q llama-index

from llama_index.core import VectorStoreIndex, Document

# ------------------------------------------
# STEP 1: Create multiple documents
# ------------------------------------------

documents = [

    Document(
        text="""
        Python is a programming language.
        It is widely used for artificial intelligence,
        machine learning, automation and data science.
        Python is known for its simple syntax.
        """
    ),

    Document(
        text="""
        Haystack is a framework for building AI applications.
        It can be used for search, document retrieval,
        question answering and RAG systems.
        """
    ),

    Document(
        text="""
        LlamaIndex is a framework for connecting
        large language models with external data.
        It provides tools for indexing and querying documents.
        """
    ),

    Document(
        text="""
        RAG stands for Retrieval Augmented Generation.
        A RAG system retrieves relevant information
        from documents and gives that information
        to a language model to generate an answer.
        """
    )
]

print("Number of documents:", len(documents))

# ------------------------------------------
# STEP 2: Create index
# ------------------------------------------

index = VectorStoreIndex.from_documents(documents)

print("Multi-document index created!")

# ------------------------------------------
# STEP 3: Create query engine
# ------------------------------------------

query_engine = index.as_query_engine()

# ------------------------------------------
# STEP 4: Ask questions
# ------------------------------------------

questions = [
    "What is Python used for?",
    "What is Haystack?",
    "What is LlamaIndex?",
    "What does RAG mean?"
]

# ------------------------------------------
# STEP 5: Generate answers
# ------------------------------------------

for question in questions:

    print("\n================================")
    print("Question:", question)
    print("================================")

    response = query_engine.query(question)

    print("Answer:")
    print(response)

print("\nMulti-document RAG project completed!")

Number of documents: 4
Multi-document index created!

Question: What is Python used for?
Answer:
Context information is below.
---------------------
Python is a programming language.
        It is widely used for artificial intelligence,
        machine learning, automation and data science.
        Python is known for its simple syntax.

Haystack is a framework for building AI applications.
        It can be used for search, document retrieval,
        question answering and RAG systems.
---------------------
Given the context information and not prior knowledge, answer the query.
Query: What is Python used for?
Answer: 

Question: What is Haystack?
Answer:
Context information is below.
---------------------
Haystack is a framework for building AI applications.
        It can be used for search, document retrieval,
        question answering and RAG systems.

LlamaIndex is a framework for connecting
        large language models with external data.
        It provides tools for indexi